# Causal Masking

AI & ML Foundations: Attention and Transformers · *Mathematics for AI and Machine Learning* by Fuheng Wu

Hide future tokens with a -inf mask and see each row of attention renormalize over the past only.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/causal-mask/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(4)

# The demo's setup: 6 tokens, width 8, one attention head
tokens = ["the", "cat", "sat", "on", "the", "mat"]
T, d = len(tokens), 8
X = torch.randn(T, d)
Wq, Wk, Wv = (torch.randn(d, d) / math.sqrt(d) for _ in range(3))


def attend(X, causal):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    S = Q @ K.T / math.sqrt(d)
    if causal:
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)   # True above the diagonal
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)
    return A, A @ V


A, out = attend(X, causal=True)
print("causal attention weights (row = query):")
for t, row in zip(tokens, A.tolist()):
    print(f"  {t:4s}", " ".join(f"{v:.2f}" for v in row))
print("weights above the diagonal are exactly 0:", bool((torch.triu(A, 1) == 0).all()))

# Same as PyTorch's built-in causal attention
ref = F.scaled_dot_product_attention((X @ Wq)[None], (X @ Wk)[None], (X @ Wv)[None], is_causal=True)[0]
print("matches is_causal=True:", torch.allclose(out, ref, atol=1e-6))

# ── Change token 5: outputs 1-4 must not move ───────────────────────────────
k = 4                                          # 0-based index of token 5
X2 = X.clone()
X2[k] += 2 * torch.randn(d)
for causal in (True, False):
    _, o1 = attend(X, causal)
    _, o2 = attend(X2, causal)
    change = (o1 - o2).norm(dim=-1)
    print(f"\n{'causal' if causal else 'no mask'}: change in each output when token {k + 1} changes:")
    print("  ", [round(c, 4) for c in change.tolist()])

causal attention weights (row = query):
  the  1.00 0.00 0.00 0.00 0.00 0.00
  cat  0.81 0.19 0.00 0.00 0.00 0.00
  sat  0.30 0.45 0.25 0.00 0.00 0.00
  on   0.49 0.25 0.14 0.12 0.00 0.00
  the  0.07 0.14 0.44 0.13 0.21 0.00
  mat  0.49 0.09 0.02 0.08 0.01 0.32
weights above the diagonal are exactly 0: True
matches is_causal=True: True

causal: change in each output when token 5 changes:
   [0.0, 0.0, 0.0, 0.0, 1.9719, 0.1269]

no mask: change in each output when token 5 changes:
   [0.1207, 0.5332, 0.7316, 0.7904, 1.8811, 0.1269]
